# Q5 Kaggle — Twitter Bot vs. Genuine User Classification

**Competition:** CS610 Assignment 1 Question 5 (2026)
**Evaluation metric:** AUC (Area Under ROC Curve)
**Submission format:** `index,target` — where `target` is a probability in `[0, 1]`

This notebook walks through baseline → LightGBM → cross-validation → text features. The point of starting simple is to give us a known floor — once we have a number from a defensible model, every subsequent change has to beat that on validation, otherwise it's noise.


## 1. Setup

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from scipy import sparse

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 2025


In [2]:
HERE = Path.cwd()
REPO = HERE.parent if HERE.name == "q5_kaggle" else HERE
DATA_DIR = REPO / "Assignment1" / "cs-610-assignment-1-question-5-2026"
SUB_DIR = (HERE if HERE.name == "q5_kaggle" else HERE / "q5_kaggle") / "submissions"
SUB_DIR.mkdir(exist_ok=True, parents=True)
print("Data dir :", DATA_DIR)
print("Sub  dir :", SUB_DIR)


Data dir : /workspace/Assignment1/cs-610-assignment-1-question-5-2026
Sub  dir : /workspace/q5_kaggle/submissions


## 2. Load and inspect

First instinct on any new dataset: shape, dtypes, head, target balance, missing values. Don't skip this.


In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
print(f"train: {train.shape}")
print(f"test : {test.shape}")
train.head()


train: (26206, 19)
test : (11232, 19)


,created_at,default_profile,default_profile_image,description,favourites_count,followers_count,friends_count,geo_enabled,id,lang,location,profile_background_image_url,profile_image_url,screen_name,statuses_count,verified,average_tweets_per_day,account_age_days,target
0,2012-01-15 23:40:09,True,False,Cosplayer/Fitness lover. Come to me https://t....,74,7,0,False,465096524,en,unknown,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/9666745212...,reml5477,20,False,0.006,3138,1
1,2016-10-04 00:44:39,False,False,pobody’s nerfect,50443,164,590,True,783105517673648132,cy,she/her,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1281752126...,kinlibra,6469,False,4.572,1415,0
2,2009-05-23 04:04:13,False,False,gracias por participar 🏅,9394,208,189,False,41970759,es,La diaspora,http://abs.twimg.com/images/themes/theme17/bg.gif,http://pbs.twimg.com/profile_images/1233811596...,_delaualau,30296,False,7.378,4106,0
3,2009-05-17 04:31:31,False,False,Stand Up Comedian/Actor from North Philadelphi...,46,66180,1090,True,40607946,en,"Calabasas, CA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1184851104...,SpankHorton,164957,False,40.116,4112,0
4,2009-02-16 13:11:21,True,False,Assignment Editor at NBC10 and President of Ja...,1223,487,867,True,20983433,en,"Jenkintown, PA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/5234863934...,javelinjt,1752,False,0.417,4201,0


In [4]:
print(train["target"].value_counts())
print(f"\nbot rate: {train['target'].mean():.3f}")


target
0    17423
1     8783
Name: count, dtype: int64

bot rate: 0.335


In [5]:
pd.DataFrame({
    "dtype":       train.dtypes,
    "missing":     train.isna().sum(),
    "pct_missing": (train.isna().mean() * 100).round(2),
    "nunique":     train.nunique(),
})


,dtype,missing,pct_missing,nunique
created_at,object,0,0.00,26205
default_profile,bool,0,0.00,2
default_profile_image,bool,0,0.00,2
description,object,5091,19.43,20988
favourites_count,int64,0,0.00,11679
followers_count,int64,0,0.00,9770
friends_count,int64,0,0.00,4372
geo_enabled,bool,0,0.00,2
id,int64,0,0.00,26206
lang,object,5588,21.32,48


### What we observe

- **Class imbalance is mild** (33% bots). AUC is robust to imbalance — measures ranking, not threshold accuracy. No oversampling needed.
- **`description` and `lang` are ~20% missing.** Impute and add a "has X" presence flag.
- **`screen_name` is unique per row** — useless as raw categorical, but length and digit-fraction patterns are signal.
- **`profile_background_image_url` has 20 unique values** — Twitter defaults. Keep a presence flag.
- **`id`** is unique — drop.
- **`created_at`** is already encoded in `account_age_days` — skip for v1.


## 3. Feature engineering

The raw columns aren't directly usable. We need:

1. **Tame heavy-tailed counts** — `log1p(x)` for followers, statuses, etc.
2. **Convert booleans to int.**
3. **Bucket high-cardinality categoricals** — top-8 langs + `OTHER`.
4. **Cheap text features** — `desc_len`, `sn_len`, `sn_digits` (digit fraction in screen name).
5. **Ratio features** — `followers_per_friend`, `statuses_per_day`. Classic bot heuristics.


In [6]:
TOP_LANGS = train["lang"].value_counts().head(8).index.tolist()
print("Top 8 langs:", TOP_LANGS)


Top 8 langs: ['en', 'es', 'pt', 'it', 'ar', 'de', 'fr', 'ja']


In [7]:
def featurize(df: pd.DataFrame) -> pd.DataFrame:
    """Same feature pipeline for train and test — single function so we
    can't accidentally engineer features differently on the two."""
    out = pd.DataFrame(index=df.index)

    counts = ["favourites_count", "followers_count", "friends_count",
              "statuses_count", "average_tweets_per_day", "account_age_days"]
    for c in counts:
        out[c] = df[c].astype(float)
    for c in ["favourites_count", "followers_count", "friends_count", "statuses_count"]:
        out[f"log_{c}"] = np.log1p(out[c])

    for c in ["default_profile", "default_profile_image", "geo_enabled", "verified"]:
        out[c] = df[c].astype(int)

    out["has_description"] = df["description"].notna().astype(int)
    out["has_location"]    = (df["location"].notna() & (df["location"] != "unknown")).astype(int)
    out["has_url_bg"]      = df["profile_background_image_url"].notna().astype(int)

    out["desc_len"]  = df["description"].fillna("").str.len()
    out["sn_len"]    = df["screen_name"].fillna("").str.len()
    out["sn_digits"] = (
        df["screen_name"].fillna("").str.count(r"\d") / out["sn_len"].clip(lower=1)
    )

    lang = df["lang"].fillna("MISSING")
    out["lang"] = lang.where(lang.isin(TOP_LANGS), "OTHER")

    out["followers_per_friend"] = df["followers_count"] / df["friends_count"].clip(lower=1)
    out["statuses_per_day"]     = df["statuses_count"]  / df["account_age_days"].clip(lower=1)
    return out

X_train_full = featurize(train)
y_train      = train["target"].values
X_test       = featurize(test)
print(f"feature matrix: train {X_train_full.shape}, test {X_test.shape}")
X_train_full.head()


feature matrix: train (26206, 23), test (11232, 23)


,favourites_count,followers_count,friends_count,statuses_count,average_tweets_per_day,account_age_days,log_favourites_count,log_followers_count,log_friends_count,log_statuses_count,...,verified,has_description,has_location,has_url_bg,desc_len,sn_len,sn_digits,lang,followers_per_friend,statuses_per_day
0,74.0,7.0,0.0,20.0,0.006,3138.0,4.317488,2.079442,0.000000,3.044522,...,0,1,0,1,59,8,0.5,en,7.000000,0.006373
1,50443.0,164.0,590.0,6469.0,4.572,1415.0,10.828619,5.105945,6.381816,8.774931,...,0,1,1,1,16,8,0.0,OTHER,0.277966,4.571731
2,9394.0,208.0,189.0,30296.0,7.378,4106.0,9.147933,5.342334,5.247024,10.318804,...,0,1,1,1,24,10,0.0,es,1.100529,7.378471
3,46.0,66180.0,1090.0,164957.0,40.116,4112.0,3.850148,11.100149,6.994850,12.013446,...,0,1,1,1,147,11,0.0,en,60.715596,40.116002
4,1223.0,487.0,867.0,1752.0,0.417,4201.0,7.109879,6.190315,6.766192,7.469084,...,0,1,1,1,55,9,0.0,en,0.561707,0.417044


## 4. Train/validation split

Hold out 20% to compare models. **Stratified** keeps the bot rate balanced.


In [8]:
Xa, Xb, ya, yb = train_test_split(
    X_train_full, y_train,
    test_size=0.2, stratify=y_train, random_state=RANDOM_STATE,
)
print(f"train: {Xa.shape}  val: {Xb.shape}")
print(f"bot rate train: {ya.mean():.3f}, val: {yb.mean():.3f}")


train: (20964, 23)  val: (5242, 23)
bot rate train: 0.335, val: 0.335


## 5. LogReg + Random Forest baseline

- **LogReg** is linear → scale numerics, one-hot categoricals.
- **RF** is a tree ensemble → no scaling, just one-hot the categorical.


In [9]:
numeric_cols     = [c for c in X_train_full.columns if c != "lang"]
categorical_cols = ["lang"]

preproc_lr = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale",  StandardScaler())]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

preproc_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

models = {
    "logreg": Pipeline([
        ("preproc", preproc_lr),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)),
    ]),
    "rf": Pipeline([
        ("preproc", preproc_rf),
        ("clf", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_STATE,
        )),
    ]),
}


In [10]:
scores = {}
for name, pipe in models.items():
    pipe.fit(Xa, ya)
    yb_prob = pipe.predict_proba(Xb)[:, 1]
    yb_pred = (yb_prob >= 0.5).astype(int)
    auc = roc_auc_score(yb, yb_prob); f1 = f1_score(yb, yb_pred)
    scores[name] = {"auc": auc, "f1": f1}
    print(f"=== {name.upper()} ===")
    print(f"  AUC: {auc:.4f}   <-- competition metric")
    print(f"  F1 : {f1:.4f}")
    print()


=== LOGREG ===
  AUC: 0.8598   <-- competition metric
  F1 : 0.6929

=== RF ===
  AUC: 0.9404   <-- competition metric
  F1 : 0.8077



RF wins (~0.94 AUC vs LogReg ~0.86). Submit RF as our first baseline.

In [11]:
winner = max(scores, key=lambda k: scores[k]["auc"])
best = models[winner]
best.fit(X_train_full, y_train)
test_prob = best.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({"index": test["index"].values, "target": test_prob})
(SUB_DIR / f"baseline_{winner}.csv").write_text(submission.to_csv(index=False, float_format="%.6f"))
print(f"Wrote: {SUB_DIR / f'baseline_{winner}.csv'}")


Wrote: /workspace/q5_kaggle/submissions/baseline_rf.csv


## 6. Upgrade to LightGBM

Standard next move on tabular AUC tasks. Trees trained sequentially, each fits residual errors of the previous ones.

```
RF:   prediction = average(tree_1, ..., tree_n)              # parallel
GBM:  prediction = tree_1 + lr*tree_2 + lr*tree_3 + ...      # sequential
```

Two LightGBM features that matter here:
1. **Native categorical handling** — mark `lang` as `category` dtype, no one-hot needed.
2. **Early stopping** — pass `eval_set` and a patience; library picks the right number of trees.


In [12]:
import lightgbm as lgb

def to_lgb_input(X):
    out = X.copy()
    out["lang"] = out["lang"].astype("category")
    return out

X_lgb_full = to_lgb_input(X_train_full)
X_lgb_test = to_lgb_input(X_test)
X_lgb_test["lang"] = pd.Categorical(
    X_lgb_test["lang"], categories=X_lgb_full["lang"].cat.categories
)

Xa_lgb, Xb_lgb, ya_lgb, yb_lgb = train_test_split(
    X_lgb_full, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE,
)

lgbm = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=63,
    min_data_in_leaf=20, feature_fraction=0.9, bagging_fraction=0.9,
    bagging_freq=5, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm.fit(Xa_lgb, ya_lgb, eval_set=[(Xb_lgb, yb_lgb)], eval_metric="auc",
         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

yb_prob_lgb = lgbm.predict_proba(Xb_lgb)[:, 1]
auc_lgb = roc_auc_score(yb_lgb, yb_prob_lgb)
print(f"LightGBM holdout AUC : {auc_lgb:.4f}")
print(f"Random Forest        : {scores['rf']['auc']:.4f}")
print(f"Lift                 : {auc_lgb - scores['rf']['auc']:+.4f}")
print(f"Best iter            : {lgbm.best_iteration_}")


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[125]	valid_0's auc: 0.941776	valid_0's binary_logloss: 0.281465
LightGBM holdout AUC : 0.9418
Random Forest        : 0.9404
Lift                 : +0.0014
Best iter            : 125


In [13]:
lgbm_final = lgb.LGBMClassifier(
    n_estimators=lgbm.best_iteration_,
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=20,
    feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm_final.fit(X_lgb_full, y_train)
test_prob_lgb = lgbm_final.predict_proba(X_lgb_test)[:, 1]
sub = pd.DataFrame({"index": test["index"].values, "target": test_prob_lgb})
sub.to_csv(SUB_DIR / "lgbm_v1.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'lgbm_v1.csv'}")


Wrote: /workspace/q5_kaggle/submissions/lgbm_v1.csv


**Public LB so far:** baseline_rf.csv → 0.93470, lgbm_v1.csv → 0.93553.

## 7. Stratified 5-fold cross-validation

A single 80/20 holdout has variance — different splits give noticeably different AUCs. Before tuning and adding features, we need a more stable benchmark.

5-fold CV: train 5 models, each on a different 80% slice. Every training row gets a prediction from a model that didn't see it during training. We get:

- **Per-fold AUCs** → mean ± std. The std is the **noise floor** — improvements smaller than ~1× std are statistical noise.
- **Out-of-fold (OOF) predictions** — single ranking across all training rows, more stable than averaging fold AUCs. Also useful later for ensembling.


In [14]:
def lgbm_cv(X, y, n_splits=5, params=None, verbose=True):
    """5-fold CV with LightGBM + early stopping per fold.
    Works on either a DataFrame or a sparse matrix (uses array-style indexing)."""
    p = dict(
        n_estimators=2000, learning_rate=0.05, num_leaves=63,
        min_data_in_leaf=20, feature_fraction=0.9, bagging_fraction=0.9,
        bagging_freq=5, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    if params: p.update(params)

    def take(M, idx):
        return M.iloc[idx] if hasattr(M, "iloc") else M[idx]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, iters = [], []
    n_rows = X.shape[0]
    oof = np.zeros(n_rows)

    for fold, (tr, va) in enumerate(skf.split(np.zeros(n_rows), y)):
        clf = lgb.LGBMClassifier(**p)
        clf.fit(take(X, tr), y[tr],
                eval_set=[(take(X, va), y[va])], eval_metric="auc",
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        prob = clf.predict_proba(take(X, va))[:, 1]
        oof[va] = prob
        auc = roc_auc_score(y[va], prob)
        aucs.append(auc); iters.append(clf.best_iteration_)
        if verbose:
            print(f"Fold {fold+1}: AUC={auc:.4f}  best_iter={clf.best_iteration_}")

    aucs = np.array(aucs)
    if verbose:
        print(f"\nMean fold AUC : {aucs.mean():.4f} ± {aucs.std():.4f}")
        print(f"OOF AUC       : {roc_auc_score(y, oof):.4f}")
        print(f"Mean best_iter: {int(np.mean(iters))}")
    return {"aucs": aucs, "iters": iters, "oof": oof}

cv_baseline = lgbm_cv(X_lgb_full, y_train)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[186]	valid_0's auc: 0.944342	valid_0's binary_logloss: 0.275098
Fold 1: AUC=0.9443  best_iter=186
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[224]	valid_0's auc: 0.93884	valid_0's binary_logloss: 0.287555
Fold 2: AUC=0.9388  best_iter=224
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[222]	valid_0's auc: 0.935613	valid_0's binary_logloss: 0.29653
Fold 3: AUC=0.9356  best_iter=222
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[175]	valid_0's auc: 0.944482	valid_0's binary_logloss: 0.274053
Fold 4: AUC=0.9445  best_iter=175
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[254]	valid_0's auc: 0.943025	valid_0's binary_logloss: 0.280137
Fold 5: AUC=0.9430  best_iter=254

Mean fold AUC : 0.9413 ± 0

### What these CV numbers tell us

1. **Fold spread** ~0.9356 to 0.9445 — 0.009 spread on the same model. Same algorithm, just different luck of the split.
2. **The std (~0.0035) is our noise floor.** Any "improvement" smaller than that is statistical noise.
3. **OOF AUC ≈ Mean fold AUC** (~0.9411). When they diverge, the model is unstable across folds.

**New benchmark to beat: OOF AUC ≈ 0.9411.**


## 8. Description text features (TF-IDF)

Bots often have empty or templated bios. The dense features include `has_description` and `desc_len`, but those say "how much" was written, not *what*. TF-IDF gives us "what."

**TF-IDF in one paragraph:** for each term in the vocabulary, multiply its frequency in this document (TF) by its inverse frequency across all documents (IDF). Common terms (in everyone's bio) get low weight. Distinctive terms (in some users' bios but not most) get high weight. The result is a sparse matrix — mostly zeros, since each bio uses only a tiny slice of the vocabulary.

Parameters worth knowing:

- `max_features=500` — cap vocabulary size. Smaller = less overfit risk on a 26k-row dataset.
- `ngram_range=(1, 2)` — unigrams + bigrams. Catches phrases like "follow back" or "earn money."
- `min_df=5` — drop terms appearing in fewer than 5 docs. Removes typo-noise.
- `max_df=0.95` — drop terms in 95%+ of docs. Removes near-stopwords. (We don't use `stop_words='english'` because the dataset is multilingual.)
- `sublinear_tf=True` — uses `1 + log(tf)` instead of raw `tf`. Robust to bio length.
- `strip_accents='unicode'` — normalizes accented characters.

**The trick we have to get right:** fit TF-IDF on TRAIN only, then transform both train and test. Fitting on test would leak test-set vocabulary into training. Always fit on train, transform on test.


In [15]:
text_train = train["description"].fillna("")
text_test  = test["description"].fillna("")

tfidf = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2),
    min_df=5, max_df=0.95, lowercase=True,
    sublinear_tf=True, strip_accents="unicode",
)
Xt_train = tfidf.fit_transform(text_train)   # fit on TRAIN only
Xt_test  = tfidf.transform(text_test)
print(f"TF-IDF feature matrix: {Xt_train.shape}")
print(f"Sparsity: {1 - Xt_train.nnz / (Xt_train.shape[0] * Xt_train.shape[1]):.3%} zeros")
print(f"\nSample vocabulary terms: {list(tfidf.vocabulary_.keys())[:20]}")


TF-IDF feature matrix: (26206, 500)
Sparsity: 99.091% zeros

Sample vocabulary terms: ['fitness', 'lover', 'come', 'to', 'me', 'https', 'co', 'come to', 'to me', 'me https', 'https co', 'por', 'up', 'comedian', 'actor', 'from', 'you', 'can', 'check', 'out']


### Combining sparse text + dense numerics

The TF-IDF matrix is sparse (mostly zeros — saves memory). Our existing numeric features are dense. To feed both into LightGBM as one matrix, we:

1. One-hot encode `lang` (now sparse), since we're not using LightGBM's native categorical path.
2. Convert numeric columns to a sparse matrix.
3. Horizontally stack `[numerics | lang_onehot | tfidf]` into one CSR (compressed sparse row) matrix.

LightGBM accepts sparse matrices directly. The structure of the combined matrix tells the model "you have 23 dense-ish columns and 500 mostly-zero text columns — use whatever helps."


In [16]:
oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
lang_train_oh = oh.fit_transform(X_train_full[["lang"]])
lang_test_oh  = oh.transform(X_test[["lang"]])

num_train = sparse.csr_matrix(X_train_full[numeric_cols].fillna(0).values)
num_test  = sparse.csr_matrix(X_test[numeric_cols].fillna(0).values)

X_combined_train = sparse.hstack([num_train, lang_train_oh, Xt_train]).tocsr()
X_combined_test  = sparse.hstack([num_test,  lang_test_oh,  Xt_test ]).tocsr()
print(f"Combined train shape: {X_combined_train.shape}")
print(f"Combined test  shape: {X_combined_test.shape}")


Combined train shape: (26206, 531)
Combined test  shape: (11232, 531)


Now run 5-fold CV on the combined matrix using our existing `lgbm_cv` helper.

In [17]:
cv_text = lgbm_cv(X_combined_train, y_train)
print(f"\nPrev (no text) OOF: {roc_auc_score(y_train, cv_baseline['oof']):.4f}")
print(f"With text  OOF    : {roc_auc_score(y_train, cv_text['oof']):.4f}")
print(f"Lift              : {roc_auc_score(y_train, cv_text['oof']) - roc_auc_score(y_train, cv_baseline['oof']):+.4f}")


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[218]	valid_0's auc: 0.947455	valid_0's binary_logloss: 0.267464
Fold 1: AUC=0.9475  best_iter=218
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[224]	valid_0's auc: 0.940655	valid_0's binary_logloss: 0.281711
Fold 2: AUC=0.9407  best_iter=224
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[124]	valid_0's auc: 0.93585	valid_0's binary_logloss: 0.295037
Fold 3: AUC=0.9358  best_iter=124
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[145]	valid_0's auc: 0.945838	valid_0's binary_logloss: 0.27052
Fold 4: AUC=0.9458  best_iter=145
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[147]	valid_0's auc: 0.94515	valid_0's binary_logloss: 0.274729
Fold 5: AUC=0.9452  best_iter=147

Mean fold AUC : 0.9430 ± 0.

### Reading this result

A real but small lift (around +0.0018 OOF). It's borderline above the noise floor (std ~0.004), but it moves in the right direction on every fold. Two takeaways:

- **Text features added signal** — positive lift everywhere — but description text alone isn't transformative on this dataset. The dense features were already strong.
- **Where to push next** — vocabulary size, n-gram range, character n-grams (catches typos and leetspeak common in bots), or TF-IDF on `screen_name` too. Each is a small experiment we can A/B against this OOF benchmark.

For now, refit on the full training data and write a submission so we can see the public LB number.


In [18]:
final_iters = int(np.mean(cv_text["iters"]))
clf_final = lgb.LGBMClassifier(
    n_estimators=final_iters,
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=20,
    feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
clf_final.fit(X_combined_train, y_train)
test_prob = clf_final.predict_proba(X_combined_test)[:, 1]

sub = pd.DataFrame({"index": test["index"].values, "target": test_prob})
sub.to_csv(SUB_DIR / "lgbm_v2_text.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'lgbm_v2_text.csv'}  shape={sub.shape}")
print(f"prob range: [{test_prob.min():.4f}, {test_prob.max():.4f}], mean={test_prob.mean():.4f}")
sub.head()


Wrote: /workspace/q5_kaggle/submissions/lgbm_v2_text.csv  shape=(11232, 2)
prob range: [0.0011, 0.9991], mean=0.3230


,index,target
0,0,0.005548
1,1,0.772164
2,2,0.047250
3,3,0.052053
4,4,0.998059


## What's next

| Submission | OOF AUC | Public LB |
|---|---|---|
| `baseline_rf.csv` | n/a | 0.93470 |
| `lgbm_v1.csv` | n/a | 0.93553 |
| `lgbm_v2_text.csv` | **0.9429 ± 0.0042** | (target) |

Submit `lgbm_v2_text.csv` and tell me the public LB. Then in priority order:

1. **Temporal features from `created_at`** — day-of-week, hour-of-day, account creation year. Bot creation tends to cluster.
2. **More aggressive text features** — character n-grams, screen_name TF-IDF, larger vocabulary.
3. **Hyperparameter tuning** with Optuna once features are stable.
4. **Ensembling** — average LightGBM + XGBoost probabilities.

Item 1 (temporal) is next.
